# Transformações de intensidade, histogramas e contraste

Neste notebook, vamos aplicar transformações ponto a ponto, interpretar histogramas e CDFs e equalizar o contraste de imagens grayscale.

## Objetivos

- relacionar curvas de transformação com seus efeitos visuais;
- calcular histogramas grayscale e por canal;
- interpretar a CDF;
- comparar uma imagem antes e depois da equalização grayscale.

## 1. Instalação

O toolkit é instalado diretamente da branch main do repositório.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

## 2. Imports e objetos de apoio

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.image_analysis import ImageAnalysis
from dip_toolkit.modules.image_loader import ImageLoader
from dip_toolkit.modules.intensity_transformer import IntensityTransformer
from dip_toolkit.modules.visualization import Visualization

analysis = ImageAnalysis()
loader = ImageLoader()
transformer = IntensityTransformer()
visualization = Visualization()

## 3. Download e preparação de uma imagem real

A imagem é obtida pelo resolvedor da disciplina e carregada pelo `ImageLoader` em ordem BGR.

In [ ]:
image_path = download_course_image("astronaut.png")
image_bgr = loader.load_image(image_path)

figure, axis = visualization.show_image(
    image_bgr,
    channel_order="bgr",
    title="Imagem colorida",
)
plt.show()

### Conversão explícita para grayscale

A equalização desta issue aceita somente imagens grayscale uint8.

In [ ]:
image_gray = cv.cvtColor(image_bgr, cv.COLOR_BGR2GRAY)

figure, axis = visualization.show_image(
    image_gray,
    channel_order="gray",
    title="Imagem grayscale",
)
plt.show()

## 4. Rampa sintética

Uma rampa de 0 a 255 permite observar diretamente o mapeamento entrada-saída.

In [ ]:
ramp = np.arange(256, dtype=np.uint8).reshape(1, -1)

figure, axis = visualization.show_image(
    np.repeat(ramp, 40, axis=0),
    title="Rampa de intensidades",
)
plt.show()

## 5. Negativo

In [ ]:
negative_gray = transformer.negative(image_gray)

figure, axes = visualization.compare_images(
    [image_gray, negative_gray],
    ["Original", "Negativo"],
)
plt.show()

## 6. Transformação logarítmica

A transformação logarítmica expande intensidades baixas e comprime intensidades altas.

In [ ]:
log_gray = transformer.log_transform(image_gray, log_gain=4.0)

figure, axes = visualization.compare_images(
    [image_gray, log_gray],
    ["Original", "Logarítmica"],
)
plt.show()

## 7. Transformação gama

Valores de gamma menores que 1 clareiam a imagem; valores maiores que 1 a escurecem.

In [ ]:
gamma_light = transformer.gamma_transform(image_gray, gamma=0.5)
gamma_dark = transformer.gamma_transform(image_gray, gamma=2.0)

figure, axes = visualization.compare_images(
    [image_gray, gamma_light, gamma_dark],
    ["Original", "Gamma = 0,5", "Gamma = 2,0"],
)
plt.show()

## 8. Transformação linear por partes

Os pontos de controle estão na mesma faixa 0 a 255 da imagem.

In [ ]:
control_points = [(0, 0), (64, 32), (192, 224), (255, 255)]
piecewise_gray = transformer.piecewise_linear(image_gray, control_points)

figure, axes = visualization.compare_images(
    [image_gray, piecewise_gray],
    ["Original", "Linear por partes"],
)
plt.show()

## 9. Curvas das transformações

Aplicamos cada transformação à rampa para visualizar a função de mapeamento.

In [ ]:
curves = {
    "Negativo": transformer.negative(ramp),
    "Logarítmica": transformer.log_transform(ramp, log_gain=4.0),
    "Gamma = 0,5": transformer.gamma_transform(ramp, gamma=0.5),
    "Gamma = 2,0": transformer.gamma_transform(ramp, gamma=2.0),
    "Linear por partes": transformer.piecewise_linear(ramp, control_points),
}

figure, axis = plt.subplots(figsize=(8, 5))
for label, transformed_ramp in curves.items():
    axis.plot(ramp.ravel(), transformed_ramp.ravel(), label=label)
axis.set(
    title="Curvas de transformação",
    xlabel="Intensidade de entrada",
    ylabel="Intensidade de saída",
    xlim=(0, 255),
    ylim=(0, 255),
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()

## Comparação de histogramas e CDFs por transformação

Para cada transformação, comparamos sua distribuição de intensidades com a
imagem original. À esquerda estão os histogramas; à direita, as CDFs
normalizadas.

In [ ]:
def compare_distributions(original, transformed, title):
    original_histogram = analysis.compute_grayscale_histogram(
        original,
        bins=256,
        value_range=(0, 256),
    )
    transformed_histogram = analysis.compute_grayscale_histogram(
        transformed,
        bins=256,
        value_range=(0, 256),
    )

    original_cdf = analysis.compute_cdf(original_histogram.counts)
    transformed_cdf = analysis.compute_cdf(transformed_histogram.counts)
    bin_centers = (
        original_histogram.bin_edges[:-1] + original_histogram.bin_edges[1:]
    ) / 2

    figure, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(
        bin_centers,
        original_histogram.counts,
        label="Original",
    )
    axes[0].plot(
        bin_centers,
        transformed_histogram.counts,
        label="Transformada",
    )
    axes[0].set(
        title=f"{title}: histograma",
        xlabel="Intensidade",
        ylabel="Frequência",
    )

    axes[1].plot(bin_centers, original_cdf, label="Original")
    axes[1].plot(bin_centers, transformed_cdf, label="Transformada")
    axes[1].set(
        title=f"{title}: CDF",
        xlabel="Intensidade",
        ylabel="Probabilidade acumulada",
        ylim=(0, 1.02),
    )

    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend()

    figure.tight_layout()
    plt.show()
    return figure, axes

In [ ]:
distribution_comparisons = [
    ("Negativo", negative_gray),
    ("Logarítmica", log_gray),
    ("Gamma = 0,5", gamma_light),
    ("Gamma = 2,0", gamma_dark),
    ("Linear por partes", piecewise_gray),
]

for comparison_title, transformed_image in distribution_comparisons:
    compare_distributions(
        image_gray,
        transformed_image,
        comparison_title,
    )

## 10. Histograma grayscale

In [ ]:
gray_histogram = analysis.compute_grayscale_histogram(
    image_gray,
    bins=256,
    value_range=(0, 256),
)

figure, axis = visualization.plot_histogram(
    gray_histogram.counts,
    gray_histogram.bin_edges,
    title="Histograma grayscale",
    label="Original",
)
plt.show()

## 11. CDF

A CDF normalizada indica a fração acumulada de pixels até cada intensidade.

In [ ]:
gray_cdf = analysis.compute_cdf(gray_histogram.counts)
bin_centers = (gray_histogram.bin_edges[:-1] + gray_histogram.bin_edges[1:]) / 2

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(bin_centers, gray_cdf)
axis.set(
    title="CDF normalizada",
    xlabel="Intensidade",
    ylabel="Probabilidade acumulada",
    ylim=(0, 1.02),
)
axis.grid(alpha=0.25)
plt.show()

## 12. Equalização de contraste

Nesta issue, a equalização é definida somente para imagens grayscale uint8. Não equalizamos os canais coloridos separadamente.

In [ ]:
equalized_gray = transformer.equalize_grayscale(image_gray)

figure, axes = visualization.compare_images(
    [image_gray, equalized_gray],
    ["Antes da equalização", "Depois da equalização"],
)
plt.show()

## 13. Histogramas por canal

Como a imagem foi carregada pelo `ImageLoader` em BGR, declaramos explicitamente a ordem dos canais.

In [ ]:
color_histograms = analysis.compute_color_histograms(
    image_bgr,
    channel_order="bgr",
    bins=256,
    value_range=(0, 256),
)
channel_colors = {"b": "blue", "g": "green", "r": "red"}

figure, axis = plt.subplots(figsize=(8, 4))
for channel, histogram in color_histograms.items():
    visualization.plot_histogram(
        histogram.counts,
        histogram.bin_edges,
        ax=axis,
        title="Histogramas por canal BGR",
        label=channel.upper(),
        color=channel_colors[channel],
    )
plt.show()

## 14. Comparação dos histogramas e CDFs antes e depois

In [ ]:
equalized_histogram = analysis.compute_grayscale_histogram(
    equalized_gray,
    bins=256,
    value_range=(0, 256),
)

figure, axis = plt.subplots(figsize=(8, 4))
visualization.plot_histogram(
    gray_histogram.counts,
    gray_histogram.bin_edges,
    ax=axis,
    title="Antes e depois da equalização",
    label="Antes",
)
visualization.plot_histogram(
    equalized_histogram.counts,
    equalized_histogram.bin_edges,
    ax=axis,
    title="Antes e depois da equalização",
    label="Depois",
)
plt.show()

equalized_cdf = analysis.compute_cdf(equalized_histogram.counts)

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(bin_centers, gray_cdf, label="Antes")
axis.plot(bin_centers, equalized_cdf, label="Depois")
axis.set(
    title="CDFs antes e depois da equalização",
    xlabel="Intensidade",
    ylabel="Probabilidade acumulada",
    ylim=(0, 1.02),
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()

## Exercício final

Escolha novos valores de gamma e novos pontos de controle. Compare as imagens, as curvas, os histogramas e as CDFs. Explique quais regiões de intensidade foram expandidas ou comprimidas.

In [ ]:
# TODO: experimente outros parâmetros e registre suas observações.
student_gamma = 1.0
student_points = [(0, 0), (128, 128), (255, 255)]